In [104]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import seaborn as sns

import ipywidgets as widgets
from IPython.display import display

import torch
# from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

import sys

import torch
import torch.nn as nn
import segmentation_models_pytorch as smp

from torch.optim import AdamW

import importlib
import prep_segmentation

importlib.reload(prep_segmentation)

sys.path.append("../preprocessing")
from prep_segmentation import create_dataloaders


# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Project paths
PROJECT_ROOT = Path.cwd().parent  # If notebook is in notebooks/
DATA_DIR = PROJECT_ROOT / "data"
MODEL_PATH = PROJECT_ROOT / "models" / "sam_vit_b_01ec64.pth"

# Configuration
CONFIG = {
    "architecture": "Unet",
    "encoder": "resnet34",
    "encoder_weights": "imagenet",
    "image_size": (512, 384),
    "batch_size": 8,
    "epochs": 50,
    "learning_rate": 1e-4,
    "random_seed": 42,
}

# Reproducibility
torch.manual_seed(CONFIG["random_seed"])
np.random.seed(CONFIG["random_seed"])


Using device: cpu


In [ ]:
df = pd.read_csv(DATA_DIR / "final_data.csv")

# Convert paths back to Path objects
df["path"] = df["path"].apply(Path)
df["path_segmentation"] = df["path_segmentation"].apply(lambda x: Path(x) if pd.notna(x) else None)

In [106]:
def explore_images(df, data_dir=DATA_DIR):
    """
    Interactive image explorer for the skin lesion dataset.

    Filters:
        dataset
        age
        sex
        skin_tone_class
        diagnosis_confirm_type

    Additionally, select one of the 11 diagnoses.
    Only images with the selected diagnosis == 1 are shown.
    """

    from IPython.display import display, clear_output

    categories = ["AKIEC", "BCC", "BEN_OTH", "BKL", "DF", "INF", "MAL_OTH", "MEL", "NV", "SCCKA", "VASC"]

    # Create dropdown options, including an "All" option
    def get_options(column):
        values = df[column].dropna().unique().tolist()
        values = sorted(values, key=str)
        return [("All", None)] + [(str(v), v) for v in values]

    # Filters
    dataset = widgets.Dropdown(options=get_options("dataset"), description="Dataset:")

    age = widgets.Dropdown(options=get_options("age"), description="Age:")

    sex = widgets.Dropdown(options=get_options("sex"), description="Sex:")

    skin_tone = widgets.Dropdown(options=get_options("skin_tone_class"), description="Skin tone:")

    confirm_type = widgets.Dropdown(options=get_options("diagnosis_confirm_type"), description="Confirm type:")

    diagnosis = widgets.Dropdown(options=categories, value="MEL", description="Diagnosis:")

    n_images = widgets.IntSlider(value=10, min=1, max=50, step=1, description="Images:")

    # Output area
    output = widgets.Output()

    def update_images(*args):

        with output:
            clear_output(wait=True)

            filtered = df.copy()

            # Apply filters
            filters = {
                "dataset": dataset.value,
                "age": age.value,
                "sex": sex.value,
                "skin_tone_class": skin_tone.value,
                "diagnosis_confirm_type": confirm_type.value,
            }

            for column, value in filters.items():
                if value is not None:
                    filtered = filtered[filtered[column] == value]

            # Filter by selected diagnosis
            filtered = filtered[filtered[diagnosis.value] == 1]

            print(f"Matching images: {len(filtered)}")

            if filtered.empty:
                print("No images match the selected filters.")
                return

            # Randomly sample images
            sample = filtered.sample(n=min(n_images.value, len(filtered)))

            # Display images in a 5-column grid
            sample = filtered.sample(n=min(n_images.value, len(filtered)))

            image_widgets = []

            for _, row in sample.iterrows():
                img_path = Path(row["path"])

                if not img_path.is_absolute():
                    img_path = Path(data_dir) / img_path

                if img_path.exists():
                    img = Image.open(img_path)
                    img.thumbnail((150, 150))

                    image_widgets.append(
                        widgets.VBox(
                            [
                                widgets.HTML(f"<b>{row['image_id']}</b><br>{row['dataset']} | Age: {row['age']}"),
                                widgets.Image(value=img.convert("RGB")._repr_png_(), format="png", width=150, height=150),
                            ]
                        )
                    )

                else:
                    image_widgets.append(widgets.HTML(f"Image not found: {img_path}"))

            # Arrange in rows of five
            grid = widgets.GridBox(image_widgets, layout=widgets.Layout(grid_template_columns="repeat(5, 160px)", grid_gap="10px"))

            display(grid)

    # Update whenever a filter changes
    controls = [dataset, age, sex, skin_tone, confirm_type, diagnosis, n_images]

    for control in controls:
        control.observe(update_images, names="value")

    # Initial display
    display(
        widgets.VBox(
            [
                widgets.HTML("<h3>Skin Lesion Image Explorer</h3>"),
                widgets.HBox([dataset, age]),
                widgets.HBox([sex, skin_tone]),
                widgets.HBox([confirm_type, diagnosis]),
                n_images,
                output,
            ]
        )
    )

    update_images()


In [107]:
explore_images(df)

Segmentation

In [108]:
# Train-Test Split: 70% training, 15% validation, 15% test
from sklearn.model_selection import train_test_split

# only use labeled data from ISIC2017
df_isic = df[df.dataset == "ISIC2017"]

# 70% training, 30% temporary
df_train, df_temp = train_test_split(df_isic, test_size=0.30, random_state=42, shuffle=True, stratify=df_isic["MEL"])

# Split temporary set into 15% validation and 15% test
df_val, df_test = train_test_split(df_temp, test_size=0.50, random_state=42, shuffle=True, stratify=df_temp["MEL"])

print(f"Training:   {len(df_train)} ({len(df_train) / len(df_isic):.1%})")
print(f"Validation: {len(df_val)} ({len(df_val) / len(df_isic):.1%})")
print(f"Test:       {len(df_test)} ({len(df_test) / len(df_isic):.1%})")

Training:   1222 (70.0%)
Validation: 262 (15.0%)
Test:       262 (15.0%)


In [ ]:
# u-net with loss function binary cross entropy + dice loss
# evaluation dice coefficient or IoU (intersection over union)
# visualize pred vs truth to see systematic errors


In [110]:
# create data loaders

train_loader, val_loader, test_loader = create_dataloaders(
    df_train, df_val, df_test, batch_size=CONFIG["batch_size"], image_size=CONFIG["image_size"]
)

# Inspect one batch
images, masks = next(iter(train_loader))

print("Images:", images.shape)
print("Masks:", masks.shape)

Images: torch.Size([8, 3, 384, 512])
Masks: torch.Size([8, 1, 384, 512])


In [ ]:
# Initialize model

# maybe try smp.DeepLabV3Plus later
model = smp.Unet(encoder_name=CONFIG["encoder"], encoder_weights=CONFIG["encoder_weights"], in_channels=3, classes=1)

model = model.to(device)

print(model)


Unet(
  (encoder): ResNetEncoder(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05,

In [116]:
nn.BCEWithLogitsLoss()

BCEWithLogitsLoss()

In [112]:
# Define loss and metrics

bce_loss = nn.BCEWithLogitsLoss()

dice_loss = smp.losses.DiceLoss(mode="binary", from_logits=True)


def loss_fn(pred, target):
    return bce_loss(pred, target) + dice_loss(pred, target)


def dice_score(pred, target, threshold=0.5):

    pred = torch.sigmoid(pred)
    pred = (pred > threshold).float()

    intersection = (pred * target).sum(dim=(1, 2, 3))
    denominator = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))

    dice = (2 * intersection + 1e-7) / (denominator + 1e-7)

    return dice.mean()


In [113]:
# Optimizer

optimizer = AdamW(model.parameters(), lr=CONFIG["learning_rate"])


In [114]:
# training and validation loop


def run_epoch(model, loader, optimizer=None):

    training = optimizer is not None

    if training:
        model.train()
    else:
        model.eval()

    total_loss = 0
    total_dice = 0

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            predictions = model(images)
            loss = loss_fn(predictions, masks)

            if training:
                loss.backward()
                optimizer.step()

        batch_size = images.size(0)

        total_loss += loss.item() * batch_size
        total_dice += dice_score(predictions, masks).item() * batch_size

    return (total_loss / len(loader.dataset), total_dice / len(loader.dataset))


# Training
history = {"train_loss": [], "val_loss": [], "train_dice": [], "val_dice": []}

best_val_loss = float("inf")

for epoch in range(CONFIG["epochs"]):
    train_loss, train_dice = run_epoch(model, train_loader, optimizer)

    val_loss, val_dice = run_epoch(model, val_loader)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_dice"].append(train_dice)
    history["val_dice"].append(val_dice)

    print(
        f"Epoch {epoch + 1:02d}/{CONFIG['epochs']} | "
        f"Train loss: {train_loss:.4f} | "
        f"Val loss: {val_loss:.4f} | "
        f"Train Dice: {train_dice:.4f} | "
        f"Val Dice: {val_dice:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save(model.state_dict(), "../models/unet_best.pth")


RuntimeError: expected scalar type Double but found Float

In [ ]:
# plot training progress

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["train_loss"], label="Training")
axes[0].plot(history["val_loss"], label="Validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history["train_dice"], label="Training")
axes[1].plot(history["val_dice"], label="Validation")
axes[1].set_title("Dice coefficient")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# load best model and evaluate on test set

model.load_state_dict(torch.load("../models/unet_best.pth", map_location=device))

model.eval()

test_loss, test_dice = run_epoch(model, test_loader)

print(f"Test loss: {test_loss:.4f}")
print(f"Test Dice: {test_dice:.4f}")


In [ ]:
# visualize predictions

images, masks = next(iter(test_loader))

model.eval()

with torch.no_grad():
    predictions = torch.sigmoid(model(images.to(device)))

predictions = (predictions > 0.5).cpu().numpy()

images = images.numpy()
masks = masks.numpy()


# Reverse ImageNet normalization for visualization
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

n = min(4, len(images))

fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))

for i in range(n):
    image = images[i].transpose(1, 2, 0)
    image = image * std + mean
    image = np.clip(image, 0, 1)

    axes[i, 0].imshow(image)
    axes[i, 0].set_title("Original image")

    axes[i, 1].imshow(masks[i, 0], cmap="gray")
    axes[i, 1].set_title("Ground truth")

    axes[i, 2].imshow(predictions[i, 0], cmap="gray")
    axes[i, 2].set_title("Prediction")

    for ax in axes[i]:
        ax.axis("off")

plt.tight_layout()
plt.show()
